We'll make this a separate Phase 4 notebook and keep your existing embedding/evaluation notebook untouched.

The goal of this notebook is:

Load your existing chunks + embeddings.

Completely exclude README.md.

Implement the baseline semantic retriever.

Implement hybrid retrieval (semantic + lexical).

Add a reranker over the retrieved candidates.

Evaluate both 116 short + 41 long queries.

Compare against your current baseline.

Save the final retrieval results/evaluations.

In [1]:
!pip install rank-bm25

## Imports & configuration

In [1]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path

from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi

print("Libraries loaded successfully.")

e:\Anaconda3\envs\voiceenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully.


## Paths

In [ ]:
BASE_DIR = Path(".")

CHUNKS_FILE = BASE_DIR / "chunked_data" / "chunks.jsonl"

SHORT_QUERIES_FILE = BASE_DIR / "failure_analysis_results" / "short_queries_116_retrieval_outputs.json"
LONG_QUERIES_FILE = BASE_DIR / "failure_analysis_results" / "long_queries_41_retrieval_outputs.json"

SHORT_EVAL_FILE = BASE_DIR / "evaluation_results" / "retrieval_evaluation.json"
LONG_EVAL_FILE = BASE_DIR / "evaluation_results" / "retrieval_evaluation_long.json"

print("Base directory:", BASE_DIR.resolve())

Base directory: E:\Projects\Speech AI\Data


## Load chunks

In [3]:
chunks = []

with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            chunks.append(json.loads(line))

print("Total chunks loaded:", len(chunks))

Total chunks loaded: 96


## Exclude README completely

In [6]:
original_count = len(chunks)

chunks = [
    chunk
    for chunk in chunks
    if Path(chunk.get("document", "")).name.lower() != "readme.md"
]

print("Original chunks:", original_count)
print("Chunks after README exclusion:", len(chunks))

readme_remaining = [
    c for c in chunks
    if Path(c.get("document", "")).name.lower() == "readme.md"
]

print("README chunks remaining:", len(readme_remaining))

assert len(readme_remaining) == 0

Original chunks: 104
Chunks after README exclusion: 96
README chunks remaining: 0


## Inspect chunk structure

In [4]:
print("Example chunk:")

for key, value in chunks[0].items():
    if key == "content":
        print(f"{key}: {value[:300]}...")
    else:
        print(f"{key}: {value}")

Example chunk:
chunk_id: chunk_000001
document: Continuing Education.md
file_path: Benefits and Perks\Continuing Education.md
category: Benefits and Perks
section_path: ['Continuing Education']
section_title: Continuing Education
chunk_index: 1
word_count: 43
character_count: 243
content: One of Clef’s core values is “Be better today than yesterday,” so it’s important that we support our employees’ efforts to learn, grow, and improve. These are some of the key benefits of working at Clef, and are central to our company culture....


## Prepare corpus

In [5]:
documents = [chunk["content"] for chunk in chunks]

print("Documents prepared:", len(documents))
print("Average words:",
      np.mean([len(doc.split()) for doc in documents]))

Documents prepared: 96
Average words: 120.875


## Baseline Semantic Retrieval

## Load embedding model

Use the same embedding model you used for your original evaluation.

In [6]:
EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(EMBEDDING_MODEL)

print("Embedding model loaded:", EMBEDDING_MODEL)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3292.93it/s]


Embedding model loaded: BAAI/bge-small-en-v1.5


## Load/create embeddings

In [7]:
chunk_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
)

chunk_embeddings = np.asarray(chunk_embeddings)

print("Embedding shape:", chunk_embeddings.shape)

Batches: 100%|██████████| 3/3 [00:13<00:00,  4.50s/it]

Embedding shape: (96, 384)


## Semantic retrieval function

In [8]:
def semantic_search(query, top_k=10):
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, 1):
        result = chunks[idx].copy()
        result["rank"] = rank
        result["score"] = float(scores[idx])

        results.append(result)

    return results

## Test README exclusion

In [9]:
results = semantic_search(
    "What benefits does the company provide?",
    top_k=10
)

for r in results:
    print(
        r["rank"],
        "|",
        r.get("section_path"),
        "|",
        r.get("document")
    )

1 | ['Continuing Education'] | Continuing Education.md
2 | ['Working Remotely', 'Approach'] | Working Remotely.md
3 | ['Mission Statement', 'Our mission is to empower everyone to own their identity online.'] | Mission Statement.md
4 | ['Policy Changes'] | Policy Changes.md
5 | ['Clef Core Values', "Treat others the way they'd like to be treated."] | Clef Values.md
6 | ['Employee Privacy', 'Email and Internet Privacy'] | Employee Privacy.md
7 | ['Salary and Equity Compensation', 'Salary'] | Salary and Equity Compensation.md
8 | ['Effective meetings and group work', 'Meeting Ettiquette', 'Prerequisites for successful meetings'] | Effective Meetings.md
9 | ['Continuing Education', 'Speaker Support'] | Continuing Education.md
10 | ['Employee Privacy', 'Workspace Privacy'] | Employee Privacy.md


## Part B — Hybrid Retrieval
### Create BM25 index

In [10]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

tokenized_documents = [
    tokenize(doc)
    for doc in documents
]

bm25 = BM25Okapi(tokenized_documents)

print("BM25 index created.")

BM25 index created.


## Hybrid retrieval

We'll combine:

semantic similarity
BM25 lexical similarity

In [11]:
def normalize_scores(scores):
    scores = np.asarray(scores, dtype=float)

    if scores.max() == scores.min():
        return np.zeros_like(scores)

    return (scores - scores.min()) / (
        scores.max() - scores.min()
    )


def hybrid_search(
    query,
    top_k=10,
    semantic_weight=0.7,
    lexical_weight=0.3
):
    # Semantic
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(
        query_embedding,
        chunk_embeddings
    )[0]

    # BM25
    lexical_scores = bm25.get_scores(
        tokenize(query)
    )

    # Normalize
    semantic_norm = normalize_scores(
        semantic_scores
    )

    lexical_norm = normalize_scores(
        lexical_scores
    )

    # Combined score
    final_scores = (
        semantic_weight * semantic_norm
        +
        lexical_weight * lexical_norm
    )

    top_indices = np.argsort(
        final_scores
    )[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, 1):

        result = chunks[idx].copy()

        result["rank"] = rank
        result["hybrid_score"] = float(
            final_scores[idx]
        )
        result["semantic_score"] = float(
            semantic_scores[idx]
        )
        result["bm25_score"] = float(
            lexical_scores[idx]
        )

        results.append(result)

    return results

## Test hybrid retrieval

In [12]:
test_queries = [
    "How many days of PTO do I earn per month?",
    "What is the equity vesting schedule?",
    "What is the parental leave policy?",
    "Can I work remotely?",
    "What are the company's core values?"
]

for query in test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)

    results = hybrid_search(query, top_k=5)

    for r in results:
        print(
            f"{r['rank']}. "
            f"{r.get('section_path')} | "
            f"{r.get('document')} | "
            f"{r['hybrid_score']:.4f}"
        )


QUERY: How many days of PTO do I earn per month?
1. ['Vacation and Sick Leave'] | Vacation and Sick Leave.md | 0.9592
2. ['Other Protected Absences', 'Pregnancy Disability Leave'] | Other Protected Absences.md | 0.8338
3. ['Working Remotely', 'Policies', 'Extended Remote Work', 'Extended remote work'] | Working Remotely.md | 0.7651
4. ['Working Remotely', 'Scope'] | Working Remotely.md | 0.7324
5. ['New Parent Leave'] | New Parent Leave.md | 0.7130

QUERY: What is the equity vesting schedule?
1. ['Salary and Equity Compensation', 'Salary'] | Salary and Equity Compensation.md | 1.0000
2. ['Salary and Equity Compensation', 'Salary'] | Salary and Equity Compensation.md | 0.7359
3. ['Tracking OKRs'] | Objectives and Key Results.md | 0.7319
4. ['Budgeting'] | Budgeting.md | 0.7026
5. ['Policy Changes', 'Continuing Work'] | Policy Changes.md | 0.7017

QUERY: What is the parental leave policy?
1. ['New Parent Leave'] | New Parent Leave.md | 1.0000
2. ['Other Protected Absences', 'Pregnancy D

## Part C — Reranking
## Cell 14 — Load CrossEncoder reranker

A good lightweight starting point:

In [13]:
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(RERANKER_MODEL)

print("Reranker loaded:", RERANKER_MODEL)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2140.60it/s]


Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


## Reranked retrieval

We retrieve a larger candidate pool first, then let the reranker determine the final order.

In [14]:
def reranked_search(
    query,
    candidate_k=10,
    top_k=5
):
    candidates = hybrid_search(
        query,
        top_k=candidate_k
    )

    pairs = [
        (query, candidate["content"])
        for candidate in candidates
    ]

    rerank_scores = reranker.predict(pairs)

    for candidate, score in zip(
        candidates,
        rerank_scores
    ):
        candidate["rerank_score"] = float(score)

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    final_results = candidates[:top_k]

    for rank, result in enumerate(
        final_results,
        1
    ):
        result["rank"] = rank

    return final_results

## Test reranking

In [15]:
for query in test_queries:

    print("\n" + "=" * 80)
    print("QUERY:", query)

    results = reranked_search(
        query,
        candidate_k=10,
        top_k=5
    )

    for r in results:

        print(
            f"{r['rank']}. "
            f"{r.get('section_path')} | "
            f"{r.get('document')} | "
            f"Rerank: {r['rerank_score']:.4f}"
        )


QUERY: How many days of PTO do I earn per month?
1. ['Vacation and Sick Leave'] | Vacation and Sick Leave.md | Rerank: -2.7559
2. ['Other Protected Absences', 'Pregnancy Disability Leave'] | Other Protected Absences.md | Rerank: -6.9162
3. ['Other Protected Absences', 'Jury Duty or Witness Summons'] | Other Protected Absences.md | Rerank: -8.0055
4. ['Referral Bonuses'] | Referral Bonuses.md | Rerank: -8.8570
5. ['New Parent Leave'] | New Parent Leave.md | Rerank: -9.0391

QUERY: What is the equity vesting schedule?
1. ['Salary and Equity Compensation', 'Salary'] | Salary and Equity Compensation.md | Rerank: 0.9538
2. ['Salary and Equity Compensation', 'Salary'] | Salary and Equity Compensation.md | Rerank: -6.1205
3. ['Salary and Equity Compensation'] | Salary and Equity Compensation.md | Rerank: -10.7541
4. ['One on Ones', 'Scheduling'] | One on Ones.md | Rerank: -10.9980
5. ['Budgeting'] | Budgeting.md | Rerank: -11.0137

QUERY: What is the parental leave policy?
1. ['Other Protect

## Part D — Evaluate against your existing 158 queries
### Cell 17 — Load evaluation queries

Your existing files contain the retrieval outputs, so we'll extract the query text from them.

In [16]:
with open(
    SHORT_QUERIES_FILE,
    "r",
    encoding="utf-8"
) as f:
    short_data = json.load(f)

with open(
    LONG_QUERIES_FILE,
    "r",
    encoding="utf-8"
) as f:
    long_data = json.load(f)

print("Short query records:", len(short_data))
print("Long query records:", len(long_data))

Short query records: 116
Long query records: 41


## Inspect one record

In [17]:
print(
    json.dumps(
        short_data[0],
        indent=2,
        ensure_ascii=False
    )[:3000]
)

{
  "query": "How much vacation time do employees get?",
  "expected_document": "Vacation and Sick Leave.md",
  "results": [
    {
      "rank": 1,
      "document": "Vacation and Sick Leave.md",
      "section_title": "Vacation and Sick Leave",
      "section_path": [
        "Vacation and Sick Leave"
      ],
      "similarity_score": 0.23163411021232605,
      "word_count": 155,
      "content": "Taking time off and recharging is critical to doing your best work at Clef, so in addition to the recognized Holiday List, Clef offers 3 weeks (15 days) of paid vacation every year that accrues 1.25 of a day per month of work. Employees should schedule their vacations, let the rest of the team know, and add it to their shared work calendar at least a week in advance.\n\nEmployees also accrue 1 hour of sick leave for every 30 hours of work, but cannot accrue more than 5 days of sick leave.\n\nEmployees should report vacation and sick days to the founder they report to, who will mark it in th

## Extract queries

In [18]:
short_queries = [
    {
        "query": item["query"],
        "expected_document": item["expected_document"]
    }
    for item in short_data
]

long_queries = [
    {
        "query": item["query"],
        "expected_document": item["expected_document"]
    }
    for item in long_data
]

print("Short queries:", len(short_queries))
print("Long queries:", len(long_queries))

Short queries: 116
Long queries: 41


## Run the full retrieval experiment
### Evaluate one query set

In [19]:
def run_retrieval_experiment(
    queries,
    query_type,
    top_k=5
):
    outputs = []

    for i, item in enumerate(queries, 1):

        query = item["query"]
        expected_document = item["expected_document"]

        results = reranked_search(
            query,
            candidate_k=10,
            top_k=top_k
        )

        outputs.append({
            "query_id": i,
            "query": query,
            "query_type": query_type,
            "expected_document": expected_document,
            "results": results
        })

        if i % 10 == 0:
            print(f"Processed {i}/{len(queries)}")

    return outputs

## Run 116 short queries

In [20]:
short_reranked_results = run_retrieval_experiment(
    short_queries,
    "short"
)

print(
    "Completed short queries:",
    len(short_reranked_results)
)

Processed 10/116
Processed 20/116
Processed 30/116
Processed 40/116
Processed 50/116
Processed 60/116
Processed 70/116
Processed 80/116
Processed 90/116
Processed 100/116
Processed 110/116
Completed short queries: 116


## Run 41 long queries

In [21]:
long_reranked_results = run_retrieval_experiment(
    long_queries,
    "long"
)

print(
    "Completed long queries:",
    len(long_reranked_results)
)

Processed 10/41
Processed 20/41
Processed 30/41
Processed 40/41
Completed long queries: 41


## Part F — Save the new retrieval outputs

In [22]:
retrieval_FILE = BASE_DIR / "retrieval_improvement_outputs"
retrieval_FILE.mkdir(parents=True, exist_ok=True)

In [ ]:
with open(
    retrieval_FILE / "short_queries_116_reranked_outputs.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        short_reranked_results,
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    retrieval_FILE / "long_queries_41_reranked_outputs.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        long_reranked_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved reranked retrieval outputs.")

Saved reranked retrieval outputs.
